In [ ]:
# Install Kaggle API
!pip install kaggle

# Upload your kaggle.json API key (from Kaggle account settings)
from google.colab import files
files.upload()  # upload kaggle.json

# Download dataset directly
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!kaggle competitions download -c store-sales-time-series-forecasting
!unzip store-sales-time-series-forecasting.zip

In [ ]:
!pip install lightgbm scikit-learn pandas numpy matplotlib seaborn kaggle

In [ ]:
from google.colab import files

# Upload your kaggle.json file
files.upload()  # select your kaggle.json when prompted

import os
os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# Download dataset
!kaggle competitions download -c store-sales-time-series-forecasting
!unzip -q store-sales-time-series-forecasting.zip
!ls

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
train_df = pd.read_csv("train.csv", parse_dates=["date"])
stores_df = pd.read_csv("stores.csv")
oil_df = pd.read_csv("oil.csv", parse_dates=["date"])
holidays_df = pd.read_csv("holidays_events.csv", parse_dates=["date"])

print("Train shape:", train_df.shape)
print("\nFirst 5 rows:")
train_df.head()

In [ ]:
# Merge store info
df = train_df.merge(stores_df, on="store_nbr", how="left")

# Merge oil prices
df = df.merge(oil_df, on="date", how="left")

# Fill missing oil prices
df["dcoilwtico"] = df["dcoilwtico"].fillna(method="ffill")

# Extract date features
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["quarter"] = df["date"].dt.quarter

# Encode categorical columns
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["family_encoded"] = le.fit_transform(df["family"])
df["city_encoded"] = le.fit_transform(df["city"])
df["state_encoded"] = le.fit_transform(df["state"])
df["type_encoded"] = le.fit_transform(df["type"])

# Save class names for later
family_classes = le.classes_

print("Feature engineering done!")
print("Dataset shape:", df.shape)
df.head()

In [ ]:
# Features to use
FEATURES = [
    "store_nbr", "family_encoded", "onpromotion",
    "year", "month", "day", "day_of_week",
    "week_of_year", "is_weekend", "quarter",
    "city_encoded", "state_encoded", "type_encoded",
    "cluster", "dcoilwtico"
]

TARGET = "sales"

# Drop rows with missing target
df = df.dropna(subset=[TARGET])

X = df[FEATURES]
y = df[TARGET]

# Train/Validation split (80/20)
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("Training LightGBM model...")

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=64,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

print("\n✅ Training complete!")

In [ ]:
# Predictions
y_pred = model.predict(X_val)
y_pred = np.clip(y_pred, 0, None)  # sales can't be negative

# Metrics
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
rmsle = np.sqrt(mean_squared_error(np.log1p(y_val), np.log1p(y_pred)))

print(f"📊 Evaluation Results:")
print(f"   MAE   (Mean Absolute Error):  {mae:.4f}")
print(f"   RMSE  (Root Mean Sq Error):   {rmse:.4f}")
print(f"   RMSLE (Log Scale Error):      {rmsle:.4f}")

In [ ]:
# Feature importance
plt.figure(figsize=(10, 6))
lgb.plot_importance(model, max_num_features=15, figsize=(10, 6))
plt.title("Top 15 Most Important Features")
plt.tight_layout()
plt.savefig("feature_importance.png")
plt.show()

# Actual vs Predicted
plt.figure(figsize=(12, 5))
plt.plot(y_val.values[:200], label="Actual Sales", color="blue")
plt.plot(y_pred[:200], label="Predicted Sales", color="orange", linestyle="--")
plt.title("Actual vs Predicted Sales (First 200 samples)")
plt.legend()
plt.tight_layout()
plt.savefig("actual_vs_predicted.png")
plt.show()

In [ ]:
import joblib
from google.colab import drive

# Save model locally
joblib.dump(model, "sales_forecast_model.pkl")
print("✅ Model saved as sales_forecast_model.pkl")

# Save to Google Drive
drive.mount("/content/drive")
import shutil
shutil.copy("sales_forecast_model.pkl", "/content/drive/MyDrive/sales_forecast_model.pkl")
shutil.copy("feature_importance.png", "/content/drive/MyDrive/feature_importance.png")
shutil.copy("actual_vs_predicted.png", "/content/drive/MyDrive/actual_vs_predicted.png")
print("✅ Everything saved to Google Drive!")